# RG3 불량 조기예측: 데이터 구조부터 다시 검증

이 노트북은 **RG3에만 집중**합니다. `labeled`만 모델 학습·교차검증에 사용합니다. `unlabeled`는 파일 구조를 확인하는 선택적 진단 셀에서만 읽으며, 통계량을 학습하거나 임계값을 고르는 데 사용하지 않습니다.

현재 데이터에서 가장 중요한 발견: RG3 train의 공정값은 모두 두 번씩 나타나고, 불량 25건은 동일 공정값을 가진 두 행 중 한 행에만 있습니다. 따라서 현재 공정값만으로 두 행의 정답을 구분할 수 없습니다.

필요 패키지: `numpy`, `pandas`, `scikit-learn`, `catboost`. 설치가 필요하면 별도 셀에서 `%pip install numpy pandas scikit-learn catboost`를 실행하세요.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', 30)
SEED = 13

## 1. 입력 경로

아래 경로만 자신의 PC에 맞게 바꾸면 됩니다. 원본 데이터나 RH/LH 식별자가 복구되면 이후 셀의 `features`에 추가하기 전에 컬럼의 의미와 검사 이전 확보 가능 시점을 확인하세요.

In [5]:
DATA_DIR = Path(r'C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋')
TRAIN_PATH = DATA_DIR / 'moldset_labeled_rg3.csv'
TEST_PATH = DATA_DIR / 'moldset_unlabeled_rg3.csv'
assert TRAIN_PATH.exists(), f'학습 파일을 찾을 수 없습니다: {TRAIN_PATH}'
train = pd.read_csv(TRAIN_PATH)
print('train:', train.shape)
display(train.head(3))

train: (1182, 26)


,Unnamed: 0,PassOrFail,Injection_Time,Filling_Time,Plasticizing_Time,Cycle_Time,Clamp_Close_Time,Cushion_Position,Plasticizing_Position,Clamp_Open_Position,Max_Injection_Speed,Max_Screw_RPM,Average_Screw_RPM,Max_Injection_Pressure,Max_Switch_Over_Pressure,Max_Back_Pressure,Average_Back_Pressure,Barrel_Temperature_1,Barrel_Temperature_2,Barrel_Temperature_3,Barrel_Temperature_4,Barrel_Temperature_5,Barrel_Temperature_6,Hopper_Temperature,Mold_Temperature_3,Mold_Temperature_4
0,1211,0,-0.029099,-1.192531,1.977737,0.249511,1.033078,2.867608,-1.755816,0.0,2.187583,0.677109,-0.421798,-2.077961,-1.184902,-1.330250,-1.306878,0.096629,-0.496175,-0.002081,0.047749,0.282727,-0.354184,0.183232,-1.393485,-1.576042
1,1212,0,-0.029099,-1.192531,1.977737,0.249511,1.033078,2.867608,-1.755816,0.0,2.187583,0.677109,-0.421798,-2.077961,-1.184902,-1.330250,-1.306878,0.096629,-0.496175,-0.002081,0.047749,0.282727,-0.354184,0.183232,-1.393485,-1.576042
2,1213,0,-0.029099,-1.192531,1.528778,0.249511,1.033078,2.075089,-0.192928,0.0,2.187583,0.677109,-0.421798,-1.421642,-0.974786,-1.019133,-0.841877,0.562844,1.442567,-0.002081,0.340262,1.850096,-0.851297,0.183232,-1.535968,-1.576042


## 2. 라벨과 쌍 구조 진단

`Unnamed: 0`은 행 식별자입니다. `Clamp_Open_Position`은 RG3 train에서 상수라 모델 입력에서 제외합니다. 동일 공정값의 행을 하나의 쌍으로 묶어 교차검증 때 같은 fold에 둡니다.

In [6]:
assert train.PassOrFail.isin([0, 1]).all()
feature_cols = [c for c in train.columns if c not in ['Unnamed: 0', 'PassOrFail', 'Clamp_Open_Position']]
X = train[feature_cols].copy()
y = train.PassOrFail.to_numpy(dtype=int)
pair_id = X.groupby(feature_cols, sort=False, dropna=False).ngroup().to_numpy()
pair_sizes = pd.Series(pair_id).value_counts()
pair_label_sums = pd.DataFrame({'pair_id': pair_id, 'y': y}).groupby('pair_id').y.sum()
print('행 수:', len(train), '불량:', int(y.sum()), '불량률:', round(y.mean(), 4))
print('쌍 크기 분포:', pair_sizes.value_counts().sort_index().to_dict())
print('쌍의 불량 개수 분포:', pair_label_sums.value_counts().sort_index().to_dict())
print('상수 컬럼:', [c for c in train.columns if train[c].nunique(dropna=False) == 1])
assert pair_sizes.eq(2).all(), '예상과 다른 쌍 구조입니다. 데이터를 다시 확인하세요.' 

행 수: 1182 불량: 25 불량률: 0.0212
쌍 크기 분포: {2: 591}
쌍의 불량 개수 분포: {0: 566, 1: 25}
상수 컬럼: ['Clamp_Open_Position']


**해석:** 한 쌍의 두 행은 입력 공정값이 같지만, 25개 불량 쌍은 라벨이 서로 다릅니다. 이때 모델은 두 행에 동일한 점수를 줄 수밖에 없습니다. 원본의 RH/LH 또는 cavity 정보가 중요합니다.

In [7]:
pair_members = pd.DataFrame({'pair_id': pair_id, 'row': np.arange(len(train)), 'id': train['Unnamed: 0'], 'y': y})
pair_members['order_in_pair'] = pair_members.groupby('pair_id').cumcount()
print('쌍 안의 순서별 불량 수:')
display(pair_members.groupby('order_in_pair').agg(rows=('y', 'size'), defects=('y', 'sum')))
display(pair_members[pair_members.y.eq(1)].head(10))

쌍 안의 순서별 불량 수:


,rows,defects
order_in_pair,,
0,591,23
1,591,2


,pair_id,row,id,y,order_in_pair
136,68,136,1347,1,0
160,80,160,1371,1,0
170,85,170,1381,1,0
204,102,204,1415,1,0
244,122,244,1455,1,0
300,150,300,1511,1,0
304,152,304,1515,1,0
310,155,310,1521,1,0
322,161,322,1533,1,0
346,173,346,1557,1,0


## 3. test 구조 확인만 수행

아래 셀은 test의 라벨을 추정하거나 모델에 맞추지 않습니다. test가 train과 같은 데이터 구조인지 확인하기 위한 읽기 전용 진단입니다. 실행을 건너뛰어도 뒤의 모델 검증은 동일합니다.

In [8]:
if TEST_PATH.exists():
    test = pd.read_csv(TEST_PATH)
    test_features = test.drop(columns=['Unnamed: 0'])
    test_group_sizes = test_features.groupby(list(test_features.columns), sort=False, dropna=False).size()
    print('test 행 수:', len(test))
    print('같은 공정값의 반복 횟수:', test_group_sizes.value_counts().sort_index().to_dict())
    print('train/test 컬럼 일치:', list(train.drop(columns=['PassOrFail']).columns) == list(test.columns))
    print('train의 Clamp_Open_Position 고유값:', train.Clamp_Open_Position.nunique(),
          'test:', test.Clamp_Open_Position.nunique())
    means = pd.DataFrame({'train_mean': train[feature_cols].mean(), 'test_mean': test[feature_cols].mean(),
                          'train_std': train[feature_cols].std(ddof=0), 'test_std': test[feature_cols].std(ddof=0)})
    display(means.head(10).round(4))
else:
    print('test 파일이 없어 이 진단을 건너뜁니다.')

test 행 수: 35941
같은 공정값의 반복 횟수: {1: 23473, 2: 6234}
train/test 컬럼 일치: True
train의 Clamp_Open_Position 고유값: 1 test: 53


,train_mean,test_mean,train_std,test_std
Injection_Time,-0.0,0.0,1.0,1.0
Filling_Time,-0.0,0.0,1.0,1.0
Plasticizing_Time,-0.0,0.0,1.0,1.0
Cycle_Time,0.0,0.0,1.0,1.0
Clamp_Close_Time,0.0,-0.0,1.0,1.0
Cushion_Position,0.0,-0.0,1.0,1.0
Plasticizing_Position,0.0,0.0,1.0,1.0
Max_Injection_Speed,-0.0,0.0,1.0,1.0
Max_Screw_RPM,-0.0,-0.0,1.0,1.0
Average_Screw_RPM,-0.0,0.0,1.0,1.0


## 4. 검증 지표와 모델

PR-AUC는 `average_precision_score`로 계산합니다. 불량이 드물기 때문에 무작위 점수의 기준은 불량 비율입니다. 검사 가능한 상위 5%, 10%, 20%에서 실제 검출 수를 함께 봅니다. 5겹 결과는 불량 25건에 민감하므로 작은 모델 차이를 확정적인 우열로 해석하지 않습니다.

In [9]:
def make_models():
    return {
        'Logistic': LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000),
        'RandomForest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
            max_features=0.7, class_weight='balanced_subsample', n_jobs=1, random_state=SEED),
        'CatBoost': CatBoostClassifier(iterations=350, depth=3, learning_rate=0.03,
            l2_leaf_reg=8, loss_function='Logloss', auto_class_weights='Balanced',
            verbose=False, thread_count=1, random_seed=SEED)
    }

def evaluate(y_true, scores):
    result = {'PR-AUC': average_precision_score(y_true, scores),
              '무작위 기준': y_true.mean()}
    for pct in (5, 10, 20):
        k = round(len(y_true) * pct / 100)
        chosen = np.argsort(scores, kind='stable')[-k:]
        result[f'상위 {pct}% 검출'] = f'{int(y_true[chosen].sum())}/{int(y_true.sum())}'
    return result

## 5. 개별 행 예측: 쌍 묶음 교차검증

동일 공정값의 두 행이 학습과 검증에 나뉘지 않도록 `StratifiedGroupKFold`를 사용합니다.

In [10]:
row_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
row_results = {}
for name, model in make_models().items():
    oof = cross_val_predict(model, X, y, cv=row_cv, groups=pair_id,
                            method='predict_proba')[:, 1]
    row_results[name] = evaluate(y, oof)
display(pd.DataFrame(row_results).T)

,PR-AUC,무작위 기준,상위 5% 검출,상위 10% 검출,상위 20% 검출
Logistic,0.018212,0.021151,0/25,0/25,4/25
RandomForest,0.030887,0.021151,3/25,3/25,3/25
CatBoost,0.021406,0.021151,1/25,3/25,3/25


## 6. 쌍 단위 위험 예측

두 제품 중 하나라도 불량이면 그 쌍을 불량으로 정의합니다. 이 과제는 개별 제품의 RH/LH 구분 문제를 제거하지만, 실제 공정값이 **불량이 발생할 쌍**을 찾아낼 수 있는지 검증합니다.

In [11]:
pair_table = train.copy()
pair_table['pair_id'] = pair_id
pair_X = pair_table.groupby('pair_id', sort=True)[feature_cols].first()
pair_y = pair_table.groupby('pair_id', sort=True).PassOrFail.max().to_numpy(dtype=int)
print('쌍 수:', len(pair_y), '불량 쌍:', int(pair_y.sum()), '기준 불량률:', round(pair_y.mean(), 4))
pair_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pair_results = {}
for name, model in make_models().items():
    oof = cross_val_predict(model, pair_X, pair_y, cv=pair_cv,
                            method='predict_proba')[:, 1]
    pair_results[name] = evaluate(pair_y, oof)
display(pd.DataFrame(pair_results).T)

쌍 수: 591 불량 쌍: 25 기준 불량률: 0.0423


,PR-AUC,무작위 기준,상위 5% 검출,상위 10% 검출,상위 20% 검출
Logistic,0.036867,0.042301,0/25,1/25,4/25
RandomForest,0.040379,0.042301,0/25,2/25,4/25
CatBoost,0.042133,0.042301,0/25,2/25,3/25


## 7. 다음 실험을 시작하기 위한 데이터 요청

1. 원본의 **RH/LH 또는 cavity 식별자**를 찾습니다. `Unnamed: 0` 홀짝을 측 식별자로 단정하면 안 됩니다.
2. 생산 시각, LOT, 설비 ID, 원래 측정값과 StandardScaler의 평균·표준편차를 확인합니다. 현재 train과 test는 파일별로 별도 표준화되어 있어 동일 숫자의 물리적 의미가 달라질 수 있습니다.
3. 원본이 확보되면 쌍·LOT를 함께 묶고 미래 기간을 따로 검증합니다. 그 뒤에야 현장 검사 물량에 맞는 임계값을 결정합니다.

현재 노트북의 결과가 기준선입니다. 새 컬럼을 넣었을 때 반드시 이 기준선과 **같은 검증 규칙**으로 비교하세요.